# Ordered Weighted Average (OWA)

An **Ordered Weighted Average (OWA)** combines several numeric inputs after first **sorting the input values**. The important distinction is that an OWA weight is attached to an **ordered position** (largest, second largest, and so on), not permanently to a particular source.

## Learning objectives

By the end of this notebook, you should be able to:

1. **Describe** the difference between an OWA and a weighted average.
2. **Compute** an OWA by sorting inputs and applying a valid weight vector.
3. **Choose OWA weights** that reproduce familiar aggregation operators such as the maximum, minimum, mean, and median.
4. **Interpret** how changing the OWA weights changes the behavior of the aggregation.
5. **Estimate OWA weights from examples** using a least-squares / pseudoinverse solution.

## What to pay attention to

As you run the notebook, keep asking: **What does each weight mean after the inputs have been sorted?** That idea is the key to understanding OWA operators.


## 1. OWA notation

Suppose we have $N$ sources with real-valued inputs:

- Sources: $X = \{x_1,x_2,\ldots,x_N\}$
- Input values: $h(x_i) \in \mathbb{R}$, abbreviated as $h_i=h(x_i)$
- Input vector: $\mathbf{h}^T=(h_1,h_2,\ldots,h_N)$

Before applying the weights, sort the inputs from largest to smallest. We use $\pi$ to describe this ordering:

$$h_{\pi(1)} \ge h_{\pi(2)} \ge \cdots \ge h_{\pi(N)}.$$

The OWA weight vector is

$$\mathbf{w}=(w_1,w_2,\ldots,w_N)^T,$$

where each $w_i \in [0,1]$ and

$$\sum_{i=1}^{N}w_i=1.$$

The OWA is then

$$\operatorname{OWA}_{\mathbf{w}}(\mathbf{h})
=\sum_{i=1}^{N} w_i h_{\pi(i)}
=\mathbf{w}^T\mathbf{h}_{\pi}.$$

### Key idea

The first weight $w_1$ multiplies the **largest input**, $w_2$ multiplies the **second-largest input**, and so on. If the values change, the source receiving a particular weight may also change. This is what makes an OWA different from a standard weighted average.


## 2. Compute an OWA by hand and in Python

We will start with four inputs and equal weights. Before running the next cell, predict the result.

Because all four weights are $1/4$, what familiar aggregation operator should this OWA reproduce?


In [ ]:
import numpy as np

# Four source values. N is inferred from the data so the example is easy to modify.
h = np.asarray([0.5, 0.4, 0.9, 0.1], dtype=float)
N = len(h)

# OWA weights correspond to RANKED positions, not to x1, x2, ... directly.
# Equal weights make this OWA equivalent to the arithmetic mean.
w = np.ones(N) / N

# A valid OWA weight vector should contain nonnegative values that sum to 1.
assert np.all(w >= 0), "OWA weights must be nonnegative."
assert np.isclose(w.sum(), 1.0), "OWA weights must sum to 1."

# Step 1: sort the input values from largest to smallest.
h_sorted = np.sort(h)[::-1]

# Step 2: multiply each ordered value by its corresponding OWA weight and sum.
y = np.dot(w, h_sorted)

print("Original input: ", h)
print("Sorted input:   ", h_sorted)
print("OWA weights:    ", w)
print("OWA result:     ", y)

### Check your understanding

With equal weights, sorting does not change the final arithmetic mean, so the result above should equal `np.mean(h)`. The sorting step still matters conceptually because other OWA weight vectors emphasize particular **ranks**.

Try changing only `w` in the cell above. Can you make the OWA return each of the following?

- **Maximum** of the inputs
- **Minimum** of the inputs
- **Median** of the inputs

For $N=4$, think carefully about the median: NumPy defines it as the average of the two middle values. What OWA weights reproduce that definition?

### A note about the mode

The **mode** is different. An OWA is a weighted sum of ordered numeric values, while the mode depends on how often values occur. In general, there is no single fixed OWA weight vector that computes the statistical mode for arbitrary inputs. Try to explain why before moving on.


## 3. Learning OWA weights from examples

So far we selected the weights ourselves. We can also ask a different question:

> If we are given many input examples and the desired output for each example, can we recover OWA weights that imitate that aggregation rule?

After sorting every training example, an OWA is linear in the weights:

$$\mathbf{y} \approx X_{\text{sorted}}\mathbf{w}.$$

A least-squares solution can therefore be computed with the **Moore–Penrose pseudoinverse**:

$$\hat{\mathbf{w}} = X_{\text{sorted}}^{+}\mathbf{y}.$$

The next cell generates synthetic examples for a known aggregation rule and then tries to recover its weights. We use a fixed random-number seed so everyone gets the same result when running the notebook.


In [ ]:
import numpy as np

# Reproducible random-number generator: students should see the same example each run.
rng = np.random.default_rng(42)

M = 25  # number of training examples
N = 3   # number of inputs in each example

# Generate M examples, each containing N values in [0, 1).
X = rng.random((M, N))

# OWA operates on ordered values, so sort EACH ROW from largest to smallest.
X_sorted = np.sort(X, axis=1)[:, ::-1]

# Choose the target aggregation rule.
# Try "max", "mean", or "median" and rerun the cell.
target = "mean"

if target == "max":
    y = np.max(X_sorted, axis=1)
    expected_w = np.array([1.0, 0.0, 0.0])
elif target == "mean":
    y = np.mean(X_sorted, axis=1)
    expected_w = np.ones(N) / N
elif target == "median":
    # N=3 here, so the median is the middle ordered value.
    y = X_sorted[:, N // 2]
    expected_w = np.array([0.0, 1.0, 0.0])
else:
    raise ValueError("target must be 'max', 'mean', or 'median'")

# pinv finds a least-squares solution to X_sorted @ w ≈ y.
X_pinv = np.linalg.pinv(X_sorted)
learned_w = X_pinv @ y

print("Target operator:", target)
print("Learned weights: ", np.round(learned_w, 6))
print("Expected weights:", expected_w)
print("Sum of learned weights:", learned_w.sum())


### What should you learn from this result?

For these noise-free synthetic examples, the learned weights should be very close to the weights we expected. That demonstrates an important connection: **once the inputs are sorted, learning an OWA can be treated as a linear least-squares problem.**

There is also an important limitation. The pseudoinverse solves the least-squares problem, but it does **not automatically enforce the OWA constraints** $w_i\ge 0$ and $\sum_i w_i=1$. In this clean example the recovered solution should satisfy them approximately, but a real learning problem may require constrained optimization.

### Experiments

Change `target` to `"max"` and `"median"`. For each run, compare `learned_w` with `expected_w` and explain why the location of the largest weight makes sense.

Then increase or decrease `M`. Does the pseudoinverse still recover the expected weights? What happens if you deliberately add noise to `y`?

## Takeaways

You should now be able to explain that an OWA (1) sorts the inputs, (2) weights **ordered positions**, and (3) sums the weighted values. Different weight vectors produce different aggregation behavior, and OWA weights can also be estimated from input/output examples using least squares.

**Before you leave this notebook:** make sure you can explain, in your own words, why `[1, 0, 0]` computes the maximum when the inputs are sorted in descending order—and why that same vector would mean something different if we skipped the sorting step.
